# Notebook 2 — Nettoyage et transformation des données

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StringType,
    DateType,
    DoubleType,
    IntegerType,
)

spark = (
    SparkSession.builder
    .appName("TradeCorp - Nettoyage")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

DATA_PATH = "/home/jovyan/data"
TMP_PATH = "/home/jovyan/data/tmp"

def lire_csv(nom_fichier):
    return (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(f"{DATA_PATH}/{nom_fichier}")
    )

df_customers = lire_csv("customers.csv")
df_orders = lire_csv("orders.csv")
df_order_details = lire_csv("order_details.csv")
df_products = lire_csv("products.csv")
df_categories = lire_csv("categories.csv")
df_suppliers = lire_csv("suppliers.csv")
df_employees = lire_csv("employees.csv")
df_shippers = lire_csv("shippers.csv")

dataframes = {
    "customers": df_customers,
    "orders": df_orders,
    "order_details": df_order_details,
    "products": df_products,
    "categories": df_categories,
    "suppliers": df_suppliers,
    "employees": df_employees,
    "shippers": df_shippers,
}

print("DataFrames chargés :", len(dataframes))

DataFrames chargés : 8


## Q11 — Comptage des valeurs nulles

In [2]:
for nom, df in dataframes.items():
    print(f"\n===== VALEURS NULLES : {nom} =====")

    resultats = []

    for colonne in df.columns:
        nombre_nulls = (
            df.filter(F.col(colonne).isNull())
            .count()
        )

        resultats.append((colonne, nombre_nulls))

    df_nulls = spark.createDataFrame(
        resultats,
        ["colonne", "valeurs_nulles"]
    )

    df_nulls.show(len(resultats), truncate=False)


===== VALEURS NULLES : customers =====
+-------------+--------------+
|colonne      |valeurs_nulles|
+-------------+--------------+
|customer_id  |0             |
|company_name |0             |
|contact_name |0             |
|contact_title|0             |
|address      |0             |
|city         |0             |
|region       |60            |
|postal_code  |1             |
|country      |0             |
|phone        |0             |
|fax          |22            |
+-------------+--------------+


===== VALEURS NULLES : orders =====
+----------------+--------------+
|colonne         |valeurs_nulles|
+----------------+--------------+
|order_id        |0             |
|customer_id     |0             |
|employee_id     |0             |
|order_date      |0             |
|required_date   |0             |
|shipped_date    |21            |
|ship_via        |0             |
|freight         |0             |
|ship_name       |0             |
|ship_address    |0             |
|ship_city     

## Q12 — Traitement des valeurs nulles

In [3]:
nombre_orders_avant = df_orders.count()

df_orders_clean = df_orders.dropna(
    subset=["shipped_date"]
)

nombre_orders_apres = df_orders_clean.count()

print("Commandes avant nettoyage :", nombre_orders_avant)
print("Commandes après nettoyage :", nombre_orders_apres)
print(
    "Commandes supprimées :",
    nombre_orders_avant - nombre_orders_apres
)

Commandes avant nettoyage : 830
Commandes après nettoyage : 809
Commandes supprimées : 21


In [4]:
medianes = df_products.approxQuantile(
    "unit_price",
    [0.5],
    0.01
)

mediane_unit_price = medianes[0] if medianes else 0.0

df_products_clean = df_products.fillna(
    {"unit_price": mediane_unit_price}
)

print("Médiane de unit_price :", mediane_unit_price)

df_products_clean.filter(
    F.col("unit_price").isNull()
).show()

Médiane de unit_price : 19.45
+----------+------------+-----------+-----------+-----------------+----------+--------------+--------------+-------------+------------+
|product_id|product_name|supplier_id|category_id|quantity_per_unit|unit_price|units_in_stock|units_on_order|reorder_level|discontinued|
+----------+------------+-----------+-----------+-----------------+----------+--------------+--------------+-------------+------------+
+----------+------------+-----------+-----------+-----------------+----------+--------------+--------------+-------------+------------+



## Q13 — Conversion des types

In [5]:
colonnes_dates_orders = [
    "order_date",
    "required_date",
    "shipped_date",
]

for colonne in colonnes_dates_orders:
    df_orders_clean = df_orders_clean.withColumn(
        colonne,
        F.to_date(F.col(colonne), "yyyy-MM-dd")
    )

df_orders_clean.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- employee_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- required_date: date (nullable = true)
 |-- shipped_date: date (nullable = true)
 |-- ship_via: integer (nullable = true)
 |-- freight: double (nullable = true)
 |-- ship_name: string (nullable = true)
 |-- ship_address: string (nullable = true)
 |-- ship_city: string (nullable = true)
 |-- ship_region: string (nullable = true)
 |-- ship_postal_code: string (nullable = true)
 |-- ship_country: string (nullable = true)



In [6]:
df_order_details_clean = (
    df_order_details
    .withColumn(
        "unit_price",
        F.col("unit_price").cast(DoubleType())
    )
    .withColumn(
        "quantity",
        F.col("quantity").cast(IntegerType())
    )
    .withColumn(
        "discount",
        F.col("discount").cast(DoubleType())
    )
)

df_order_details_clean.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- discount: double (nullable = true)



## Q14 — Nettoyage des chaînes de caractères

In [7]:
df_customers_clean = df_customers

colonnes_texte_customers = [
    champ.name
    for champ in df_customers.schema.fields
    if isinstance(champ.dataType, StringType)
]

for colonne in colonnes_texte_customers:
    df_customers_clean = df_customers_clean.withColumn(
        colonne,
        F.trim(F.col(colonne))
    )

df_customers_clean = (
    df_customers_clean
    .withColumn(
        "contact_name",
        F.initcap(F.col("contact_name"))
    )
    .withColumn(
        "country",
        F.upper(F.col("country"))
    )
)

df_customers_clean.select(
    "customer_id",
    "contact_name",
    "country"
).show(10, truncate=False)

+-----------+------------------+-------+
|customer_id|contact_name      |country|
+-----------+------------------+-------+
|ALFKI      |Maria Anders      |GERMANY|
|ANATR      |Ana Trujillo      |MEXICO |
|ANTON      |Antonio Moreno    |MEXICO |
|AROUT      |Thomas Hardy      |UK     |
|BERGS      |Christina Berglund|SWEDEN |
|BLAUS      |Hanna Moos        |GERMANY|
|BLONP      |Frédérique Citeaux|FRANCE |
|BOLID      |Martín Sommer     |SPAIN  |
|BONAP      |Laurence Lebihan  |FRANCE |
|BOTTM      |Elizabeth Lincoln |CANADA |
+-----------+------------------+-------+
only showing top 10 rows


## Q15 — Renommage des colonnes

In [8]:
df_order_details_clean = (
    df_order_details_clean
    .withColumnRenamed("unit_price", "prix_unitaire")
    .withColumnRenamed("quantity", "quantite")
)

df_orders_clean = df_orders_clean.withColumnRenamed(
    "ship_via",
    "shipper_id"
)

print("Colonnes order_details :")
print(df_order_details_clean.columns)

print("\nColonnes orders :")
print(df_orders_clean.columns)

Colonnes order_details :
['order_id', 'product_id', 'prix_unitaire', 'quantite', 'discount']

Colonnes orders :
['order_id', 'customer_id', 'employee_id', 'order_date', 'required_date', 'shipped_date', 'shipper_id', 'freight', 'ship_name', 'ship_address', 'ship_city', 'ship_region', 'ship_postal_code', 'ship_country']


## Q16 — Création de la colonne sous_total

In [9]:
df_order_details_clean = df_order_details_clean.withColumn("sous_total",F.round(F.col("prix_unitaire")*F.col("quantite")*(F.lit(1.0)-F.col("discount")),2))
df_order_details_clean.select("order_id",
                              "product_id",
                              "prix_unitaire",
                              "quantite",
                              "discount",
                              "sous_total"
                             ).show(10,truncate=False)

+--------+----------+-------------+--------+--------+----------+
|order_id|product_id|prix_unitaire|quantite|discount|sous_total|
+--------+----------+-------------+--------+--------+----------+
|10248   |11        |14.0         |12      |0.0     |168.0     |
|10248   |42        |9.8          |10      |0.0     |98.0      |
|10248   |72        |34.8         |5       |0.0     |174.0     |
|10249   |14        |18.6         |9       |0.0     |167.4     |
|10249   |51        |42.4         |40      |0.0     |1696.0    |
|10250   |41        |7.7          |10      |0.0     |77.0      |
|10250   |51        |42.4         |35      |0.15    |1261.4    |
|10250   |65        |16.8         |15      |0.15    |214.2     |
|10251   |22        |16.8         |6       |0.05    |95.76     |
|10251   |57        |15.6         |15      |0.05    |222.3     |
+--------+----------+-------------+--------+--------+----------+
only showing top 10 rows


## Q17 - Création des colonnes conditionnelles

In [10]:
# Produits en stock
df_products_clean = df_products_clean.withColumn(
    "en_stock",
    F.col("units_in_stock")>0
)
df_products_clean.select(
    "product_id",
    "product_name",
    "units_in_stock",
    "en_stock"
).show(10,truncate=False)

+----------+-------------------------------+--------------+--------+
|product_id|product_name                   |units_in_stock|en_stock|
+----------+-------------------------------+--------------+--------+
|1         |Chai                           |39            |true    |
|2         |Chang                          |17            |true    |
|3         |Aniseed Syrup                  |13            |true    |
|4         |Chef Anton's Cajun Seasoning   |53            |true    |
|5         |Chef Anton's Gumbo Mix         |0             |false   |
|6         |Grandma's Boysenberry Spread   |120           |true    |
|7         |Uncle Bob's Organic Dried Pears|15            |true    |
|8         |Northwoods Cranberry Sauce     |6             |true    |
|9         |Mishi Kobe Niku                |29            |true    |
|10        |Ikura                          |31            |true    |
+----------+-------------------------------+--------------+--------+
only showing top 10 rows


In [11]:
# Commandes expédiées
df_orders_clean = df_orders_clean.withColumn(
    "is_shipped",
    F.col("shipped_date").isNotNull()
)
df_orders_clean.select(
    "order_id",
    "shipped_date",
    "is_shipped"
).show(10, truncate=False)

+--------+------------+----------+
|order_id|shipped_date|is_shipped|
+--------+------------+----------+
|10248   |1996-07-16  |true      |
|10249   |1996-07-10  |true      |
|10250   |1996-07-12  |true      |
|10251   |1996-07-15  |true      |
|10252   |1996-07-11  |true      |
|10253   |1996-07-16  |true      |
|10254   |1996-07-23  |true      |
|10255   |1996-07-15  |true      |
|10256   |1996-07-17  |true      |
|10257   |1996-07-22  |true      |
+--------+------------+----------+
only showing top 10 rows


### Observation

La colonne `is_shipped` vaut toujours `True` dans le DataFrame nettoyé,
car les commandes dont `shipped_date` était nulle ont été supprimées à la
question Q12.

## Q18 — Recherche et suppression des doublons

In [12]:
nombre_total_customers = df_customers_clean.count()

In [14]:
nombre_customers_distincts = (
    df_customers_clean.select("customer_id").distinct().count())

In [16]:
nombre_doublons = (nombre_total_customers - nombre_customers_distincts)

In [19]:
print("Nombre total de clients :", nombre_total_customers)
print(
    "Nombre d'identifiants distincts :",
    nombre_customers_distincts
)
print("Nombre de doublons :", nombre_doublons)

Nombre total de clients : 91
Nombre d'identifiants distincts : 91
Nombre de doublons : 0


In [20]:
df_customers_clean = (
    df_customers_clean
    .dropDuplicates(["customer_id"])
)

print(
    "Nombre de clients après suppression :",
    df_customers_clean.count()
)

Nombre de clients après suppression : 91


## Q19 — Filtrage des commandes et des produits

In [21]:
df_orders_1997 = df_orders_clean.filter(F.year(F.col("order_date"))==1997)
print("Nombre de commandes de 1997 :", df_orders_1997.count())
df_orders_1997.select("order_id", "order_date").show(10)

Nombre de commandes de 1997 : 408
+--------+----------+
|order_id|order_date|
+--------+----------+
|   10400|1997-01-01|
|   10401|1997-01-01|
|   10402|1997-01-02|
|   10403|1997-01-03|
|   10404|1997-01-03|
|   10405|1997-01-06|
|   10406|1997-01-07|
|   10407|1997-01-07|
|   10408|1997-01-08|
|   10409|1997-01-09|
+--------+----------+
only showing top 10 rows


In [22]:
# Produits actifs et disponibles
df_products_actifs = df_products_clean.filter((F.col("units_in_stock")>0) & (F.col("discontinued").cast("int")==0))
print("Nombre de produits actifs et en stock : ", df_products_actifs.count())
df_products_actifs.select("product_id", "product_name","units_in_stock","discontinued").show(10, truncate=False)

Nombre de produits actifs et en stock :  66
+----------+-------------------------------+--------------+------------+
|product_id|product_name                   |units_in_stock|discontinued|
+----------+-------------------------------+--------------+------------+
|3         |Aniseed Syrup                  |13            |0           |
|4         |Chef Anton's Cajun Seasoning   |53            |0           |
|6         |Grandma's Boysenberry Spread   |120           |0           |
|7         |Uncle Bob's Organic Dried Pears|15            |0           |
|8         |Northwoods Cranberry Sauce     |6             |0           |
|10        |Ikura                          |31            |0           |
|11        |Queso Cabrales                 |22            |0           |
|12        |Queso Manchego La Pastora      |86            |0           |
|13        |Konbu                          |24            |0           |
|14        |Tofu                           |35            |0           |
+------

## Q20 — Sélection des colonnes employés

In [26]:
df_employees_clean = (df_employees.select("employee_id","first_name","last_name","title","hire_date","city","country",).withColumn("hire_date",F.to_date(F.col("hire_date"),"yyyy-MM-dd")).withColumn("full_name",F.concat_ws(" ",F.col("first_name"),F.col("last_name"))))
df_employees_clean.show(truncate=False)
df_employees_clean.printSchema()

+-----------+----------+---------+------------------------+----------+--------+-------+----------------+
|employee_id|first_name|last_name|title                   |hire_date |city    |country|full_name       |
+-----------+----------+---------+------------------------+----------+--------+-------+----------------+
|1          |Nancy     |Davolio  |Sales Representative    |1992-05-01|Seattle |USA    |Nancy Davolio   |
|2          |Andrew    |Fuller   |Vice President, Sales   |1992-08-14|Tacoma  |USA    |Andrew Fuller   |
|3          |Janet     |Leverling|Sales Representative    |1992-04-01|Kirkland|USA    |Janet Leverling |
|4          |Margaret  |Peacock  |Sales Representative    |1993-05-03|Redmond |USA    |Margaret Peacock|
|5          |Steven    |Buchanan |Sales Manager           |1993-10-17|London  |UK     |Steven Buchanan |
|6          |Michael   |Suyama   |Sales Representative    |1993-10-17|London  |UK     |Michael Suyama  |
|7          |Robert    |King     |Sales Representative 

## Écriture des DataFrames nettoyés en Parquet

In [27]:
dataframes_nettoyes = {
    "customers": df_customers_clean,
    "orders": df_orders_clean,
    "order_details": df_order_details_clean,
    "products": df_products_clean,
    "categories": df_categories,
    "suppliers": df_suppliers,
    "employees": df_employees_clean,
    "shippers": df_shippers,
}

for nom, df in dataframes_nettoyes.items():
    chemin = f"{TMP_PATH}/{nom}.parquet"

    (
        df.write
        .mode("overwrite")
        .parquet(chemin)
    )

    print(f"{nom} enregistré dans {chemin}")

customers enregistré dans /home/jovyan/data/tmp/customers.parquet
orders enregistré dans /home/jovyan/data/tmp/orders.parquet
order_details enregistré dans /home/jovyan/data/tmp/order_details.parquet
products enregistré dans /home/jovyan/data/tmp/products.parquet
categories enregistré dans /home/jovyan/data/tmp/categories.parquet
suppliers enregistré dans /home/jovyan/data/tmp/suppliers.parquet
employees enregistré dans /home/jovyan/data/tmp/employees.parquet
shippers enregistré dans /home/jovyan/data/tmp/shippers.parquet


In [28]:
# Vérifier les fichiers parquet
for nom in dataframes_nettoyes:
    chemin = f"{TMP_PATH}/{nom}.parquet"

    df_verification = spark.read.parquet(chemin)

    print(
        nom,
        "=>",
        df_verification.count(),
        "ligne(s)"
    )

customers => 91 ligne(s)
orders => 809 ligne(s)
order_details => 2155 ligne(s)
products => 77 ligne(s)
categories => 8 ligne(s)
suppliers => 29 ligne(s)
employees => 9 ligne(s)
shippers => 6 ligne(s)


In [ ]:
df_customers_clean.withColumn(
        ,
        F.trim(F.col(colonne))
    )